# 10 Collection


## Setup


In [ ]:
# Imports
import json
import sys
from collections import Counter
from pathlib import Path
import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import backends
import flags
import settings
import utils

pd.set_option('display.max_colwidth', 60)
print('Ready')

## Raw Records


In [ ]:
# Model Names
def name_model(record, source):
    known = {m['id'] for m in settings.MODELS.values()}
    said = flags.model_of(record) or ''
    if said in known:
        return said
    # Model Names
    
    stem = said.split(':')[0]
    same = [k for k in known if k.split(':')[0] == stem] if stem else []
    if len(same) == 1:
        return same[0]
    # Model Names
    
    
    
    from_file = [k for k in known if utils.model_slug(k) in source]
    return from_file[0] if len(from_file) == 1 else said


# Raw Records
def unpack(record, source):
    body = flags.body_of(record)
    model = name_model(record, source)
    known = {m['id'] for m in settings.MODELS.values()}
    provider = backends.provider_of(model) if model in known else ''
    blocked, truncated = flags.flags_of(body, record)
    field, sent, received = backends.USAGE_FIELDS.get(provider, ('', '', ''))
    usage = (body.get(field) or {}) if field else body
    text = backends.read_reply(provider, body) if provider else ''
    prompt_id, replicate = flags.key_of(record)
    # Load Data
    return {'model': model, 'prompt_id': prompt_id, 'replicate': replicate,
            'blocked': blocked, 'truncated': truncated,
            'input': usage.get(sent, 0) or 0, 'output': usage.get(received, 0) or 0,
            'words': len(str(text).split()), 'empty': not str(text).strip(),
            'file': source, 'unknown model': model not in known}


raw = []
for path in sorted(settings.BATCHES_DIR.glob('*_output.jsonl')):
    for line in path.read_text().splitlines():
        if line.strip():
            raw.append(unpack(json.loads(line), path.name))

raw = pd.DataFrame(raw)
if raw.empty:
    raise SystemExit(f'No raw provider files in {settings.BATCHES_DIR}')

# Model Names
known = {m['id'] for m in settings.MODELS.values()}
for source, group in raw.groupby('file'):
    named = group.loc[group['model'].isin(known), 'model']
    if named.empty or group['model'].isin(known).all():
        continue
    raw.loc[group.index[~group['model'].isin(known)],
            ['model', 'unknown model']] = [named.mode().iloc[0], False]
print(f'{len(raw):,} raw records from {raw["file"].nunique()} files')
stranger = raw[raw['unknown model']]
if len(stranger):
    print(f'\n{len(stranger)} records name a model that is not in the panel, so')
    print('their replies cannot be read and will count as empty:')
    display(stranger.groupby(['file', 'model']).size().rename('n').to_frame())
display(raw.groupby('file').size().rename('records').to_frame())

## Coverage


In [ ]:
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']

coverage = raw.groupby('model').agg(
    replies=('prompt_id', 'size'),
    prompts=('prompt_id', 'nunique'),
    duplicated=('prompt_id', lambda ids: len(ids) - len(set(zip(
        ids, raw.loc[ids.index, 'replicate'])))))
coverage['expected'] = wanted
coverage['missing'] = coverage['expected'] - coverage['replies']
display(coverage)

if (coverage['missing'] > 0).any():
    print('Short of a full pass. Re-running the part that owns the gap will')
    print('ask only for what is absent, so nothing already collected is repeated.')
if (coverage['duplicated'] > 0).any():
    print('Some prompt and replicate pairs appear twice, which means two raw')
    print('files hold the same pass. Move the older one out of data/batches.')

## Outcomes


In [ ]:
outcomes = raw.groupby('model').agg(
    replies=('prompt_id', 'size'),
    blocked=('blocked', lambda s: (s.astype(str).str.strip() != '').sum()),
    truncated=('truncated', 'sum'),
    empty=('empty', 'sum'))
outcomes['empty for no stated reason'] = raw.groupby('model').apply(
    lambda g: (g['empty'] & (g['blocked'].astype(str).str.strip() == '')
               & ~g['truncated']).sum(), include_groups=False)
for column in ['blocked', 'truncated', 'empty']:
    outcomes[f'{column} %'] = (outcomes[column] / outcomes['replies'] * 100).round(2)
display(outcomes)

reasons = raw[raw['blocked'].astype(str).str.strip() != '']
if len(reasons):
    print('the reasons providers gave:')
    display(reasons.groupby(['model', 'blocked']).size().rename('n').to_frame())

## Tokens And Cost


In [ ]:
usage = raw.groupby('model').agg(input=('input', 'sum'), output=('output', 'sum'),
                                 replies=('prompt_id', 'size'))
usage['output a reply'] = (usage['output'] / usage['replies']).round(0)

prices = {m['id']: m.get('price') for m in settings.MODELS.values()}
usage['cost'] = [
    round((row['input'] * prices[model]['input']
           + row['output'] * prices[model]['output']) / 1e6, 2)
    if prices.get(model) else None
    for model, row in usage.iterrows()]
display(usage)

priced = usage['cost'].dropna()
print(f'Total across the models that are billed by token: ${priced.sum():,.2f}')
print('Anything blank is on a subscription, so its tokens count against a quota')
print('rather than a bill.')

## Reply Length


In [ ]:
said = raw[~raw['empty']]
length = said.groupby('model')['words'].describe()[
    ['count', 'mean', '50%', '75%', 'max']].round(0)
length.columns = ['replies', 'mean', 'median', 'p75', 'longest']
display(length)

cap = settings.GENERATION['max_tokens']
print(f'The cap was {cap:,} tokens. The longest reply here is '
      f'{said["words"].max():,.0f} words, roughly '
      f'{said["words"].max() * 1.3:,.0f} tokens.')

## Failures


In [ ]:
failures = []
for path in sorted(settings.BATCHES_DIR.glob('*_output.log.jsonl')):
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        if record.get('error'):
            failures.append({'file': path.name, 'model': record.get('model', ''),
                             'error': str(record['error'])})

if failures:
    failures = pd.DataFrame(failures)
    failures['kind'] = failures['error'].str.split(':').str[0]
    display(failures.groupby(['model', 'kind']).size().rename('n').to_frame())
    print(f'{len(failures)} calls failed and were retried.')
else:
    print('No log files, so either nothing failed or the logs were tidied away.')
    print('A part that completes without failures deletes its own log.')

## Update Flags


In [ ]:
import flags

for path in sorted(settings.ADAPTATION_DIR.glob('*.jsonl')):
    stored = utils.read_lines(path)
    if stored.empty:
        continue
    model = str(stored['model'].iloc[0])
    before = int((stored['blocked'].astype(str).str.strip() != '').sum())
    counts = flags.apply(model, write=True)
    after = counts['blocked']
    moved = '' if after == before else f'   was {before}'
    print(f"  {model:<28} {after:>4} blocked, {counts['truncated']:>3} "
          f"truncated{moved}")

print('\nresults/adaptation now agrees with the raw files.')

## Ingest Check


In [ ]:
ingested = []
for path in sorted(settings.ADAPTATION_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        ingested.append(frame)

if not ingested:
    print(f'Nothing in {settings.ADAPTATION_DIR} yet.')
else:
    ingested = pd.concat(ingested, ignore_index=True)
    check = pd.DataFrame({
        'raw': raw.groupby('model').size(),
        'ingested': ingested.groupby('model').size()}).fillna(0).astype(int)
    check['difference'] = check['ingested'] - check['raw']
    columns = set(ingested.columns)
    stale = {'backend', 'temperature'} & columns
    if stale:
        print(f'{", ".join(sorted(stale))} still in results/adaptation. Those '
              f'describe the pass rather than the reply and are no longer '
              f'written. The write back cell above rewrites the file without '
              f'them.')
    display(check)
    if (check['difference'] != 0).any():
        print('A difference means a raw file was ingested more than once, or a')
        print('pass was ingested from a file no longer in data/batches.')